# MBTA Bus Reliability Trip-Level Linear Regression

**Course:** CS506 — Data Science Tools and Applications  
**Project:** MBTA Bus Service Equity Analysis

---

## Project Overview

This notebook investigates whether **demographic composition of bus route catchments** is associated with **trip-level service reliability**, while controlling for operational factors (route, time-of-day, day-of-week, prior delay).

### Research Question
> Are buses serving routes with higher concentrations of minority or low-income riders systematically more delayed than buses on other routes, after accounting for operational conditions?

### Response & Predictors

| Role | Variable | Type |
|---|---|---|
| Response (Y) | `delay_seconds` | continuous |
| Demographic | `pct_minority`, `pct_lowincome` | continuous |
| Operating | `hour`, `day_of_week`, `is_rush_hour`, `prev_delay` | continuous |
| Operating | `route_id` | categorical (one-hot encoded) |

> **Note:** `neighborhood` was dropped from the original feature set because ~47% of records had `Unknown` in the source data, which would have introduced substantial bias.

## 1. Setup & Imports

Standard scientific Python stack. We use `numpy` for linear algebra (we solve the normal equations directly rather than calling a regression library), `pandas` for data wrangling, and `scipy.stats.f` for the overall F-test p-value.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from scipy.stats import f as fdist


Mounted at /content/drive


In [3]:
import numpy as np
import pandas as pd
from scipy.stats import f as fdist

# Display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
np.set_printoptions(suppress=True, precision=4)

## 2. Load the Merged Trip-Level Dataset

The file `trip_level_merged.csv` was produced earlier by `merge_for_regression.py`. It already contains:

- Trip-level delay observations from MBTA AVL data
- Demographic measures (`pct_minority`, `pct_lowincome`) joined from the Title VI Rider Census
- Engineered operating features: `prev_delay` (lag-1 delay on the same trip), `day_of_week` derived from real `service_date`, `is_rush_hour` indicator, `hour` extracted from scheduled timestamp

So this notebook is purely the **modeling stage** — no further feature engineering happens here.

In [6]:
trip_path = "/content/drive/MyDrive/CS506 Project/trip_level_merged.csv"
trip_df = pd.read_csv(trip_path)

print(f"Loaded: {trip_df.shape[0]:,} rows x {trip_df.shape[1]} columns")
print(f"\nColumns: {trip_df.columns.tolist()}")
trip_df.head()

Loaded: 325,924 rows x 11 columns

Columns: ['service_date', 'route_id', 'direction_id', 'neighborhood', 'hour', 'day_of_week', 'is_rush_hour', 'prev_delay', 'pct_minority', 'pct_lowincome', 'delay_seconds']


/tmp/ipykernel_8536/4240741392.py:2: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  trip_df = pd.read_csv(trip_path)


,service_date,route_id,direction_id,neighborhood,hour,day_of_week,is_rush_hour,prev_delay,pct_minority,pct_lowincome,delay_seconds
0,2022-11-02,10,Inbound,South Boston,5,2,0,56.0,0.411511,0.756001,74.0
1,2022-06-09,10,Inbound,South Boston,5,3,0,74.0,0.411511,0.756001,89.0
2,2022-10-27,10,Inbound,South Boston,5,3,0,89.0,0.411511,0.756001,111.0
3,2022-09-02,10,Inbound,South Boston,5,4,0,111.0,0.411511,0.756001,130.0
4,2022-08-03,10,Inbound,South Boston,5,2,0,130.0,0.411511,0.756001,143.0


## 3. Specify Response & Predictors

We split predictors into two groups so we can handle them differently downstream:

- **Numeric predictors** enter the design matrix directly.
- **Categorical predictors** (`route_id`) get one-hot encoded with `drop_first=True` to avoid the dummy variable trap (the dropped level becomes the reference category absorbed by the intercept).

In [7]:
response = "delay_seconds"

numeric_predictors = [
    "pct_minority",
    "pct_lowincome",
    "hour",
    "day_of_week",
    "is_rush_hour",
    "prev_delay",
]

categorical_predictors = ["route_id"]

print(f"Response:               {response}")
print(f"Numeric predictors:     {numeric_predictors}")
print(f"Categorical predictors: {categorical_predictors}")

Response:               delay_seconds
Numeric predictors:     ['pct_minority', 'pct_lowincome', 'hour', 'day_of_week', 'is_rush_hour', 'prev_delay']
Categorical predictors: ['route_id']


## 4. Drop Rows with Missing Values

We use complete-case analysis on the columns that actually enter the model. Rows missing any of the response or predictor values get dropped OLS via the normal equations cannot accept NaNs.

In [8]:
needed = [response] + numeric_predictors + categorical_predictors

before = len(trip_df)
trip_df = trip_df.dropna(subset=needed)
after = len(trip_df)

print(f"Rows before: {before:,}")
print(f"Rows after:  {after:,}")
print(f"Dropped:     {before - after:,} ({(before-after)/before*100:.2f}%)")

Rows before: 325,924
Rows after:  325,924
Dropped:     0 (0.00%)


## 5. Build the Design Matrix

Numeric predictors are stacked horizontally with the one-hot-encoded route dummies to form **X** (n × p). The intercept column is added inside `fit_ols()` later, so X here has no leading column of ones.

In [9]:
X_numeric = trip_df[numeric_predictors].astype(float)
X_dummies = pd.get_dummies(trip_df[categorical_predictors], drop_first=True).astype(float)

X = pd.concat([X_numeric, X_dummies], axis=1).values
Y = trip_df[response].astype(float).values

print(f"Design matrix X:  n = {X.shape[0]:,}  |  p = {X.shape[1]}")
print(f"  - {len(numeric_predictors)} numeric predictors")
print(f"  - {X_dummies.shape[1]} route dummies (one route is the reference category)")
print(f"\nResponse Y:       length {len(Y):,}")
print(f"  - mean  = {Y.mean():.2f} sec")
print(f"  - std   = {Y.std():.2f} sec")
print(f"  - range = [{Y.min():.0f}, {Y.max():.0f}] sec")

Design matrix X:  n = 325,924  |  p = 41
  - 6 numeric predictors
  - 35 route dummies (one route is the reference category)

Response Y:       length 325,924
  - mean  = 204.16 sec
  - std   = 372.40 sec
  - range = [-2809, 3597] sec


## 6. Naive Baseline: Predict the Mean

Before fitting any model, we establish a **floor** that any meaningful regression must beat. The dumbest possible predictor is the mean of Y. By construction:

- $R^2 = 0$
- $\text{RMSE} = \text{SD}(Y)$
- $\text{MAE} = $ mean absolute deviation of Y from its mean

If our regression's RMSE/MAE aren't meaningfully lower than these, the predictors aren't doing real work.

In [12]:
Y_mean = Y.mean()
baseline_rmse = np.sqrt(np.mean((Y - Y_mean) ** 2))
baseline_mae  = np.mean(np.abs(Y - Y_mean))

print("NAIVE BASELINE (always predicts the mean)")
print("=" * 60)
print(f"Mean delay:  {Y_mean:.2f} sec")
print(f"R^2:         0.0000")
print(f"RMSE:        {baseline_rmse:.2f} sec")
print(f"MAE:         {baseline_mae:.2f} sec")

NAIVE BASELINE (always predicts the mean)
Mean delay:  204.16 sec
R^2:         0.0000
RMSE:        372.40 sec
MAE:         254.29 sec


## 7. OLS Helper Functions

Rather than copy-pasting the same regression math three times (full model, demographics-only, etc.), we wrap it once.

### `fit_ols(X, Y)`
Solves $\hat{\beta} = (X^T X)^{-1} X^T Y$ via `np.linalg.lstsq` (numerically stable for wide design matrices) and returns every Lecture 15 metric:

| Quantity | Formula |
|---|---|
| SSE | $\sum (y_i - \hat{y}_i)^2$ |
| SST | $(n-1) \cdot \text{Var}(Y)$ |
| SSM | SST − SSE |
| MSE | SSE / (n − p − 1) |
| MSM | SSM / p |
| F-statistic | MSM / MSE |
| $R^2$ | 1 − SSE/SST |
| $R^2_{adj}$ | 1 − MSE / (SST/(n−1)) |

### `print_results(...)`
Pretty-prints the dict returned by `fit_ols`, plus the intercept and numeric coefficients (route dummy coefficients are suppressed — there are dozens of them and they're nuisance controls, not the focus of this analysis).

In [13]:
def fit_ols(X, Y):
    """Fit OLS via normal equations and return standard regression metrics."""
    n, p = X.shape

    # Add intercept column; lstsq is numerically stable for wide X
    Xmat = np.column_stack([np.ones(n), X])
    bhat, *_ = np.linalg.lstsq(Xmat, Y, rcond=None)

    # Predictions and residuals
    Yhat  = Xmat @ bhat
    resid = Y - Yhat

    # Sums of squares
    SSE = np.sum(resid ** 2)
    SST = np.var(Y, ddof=1) * (n - 1)
    SSM = SST - SSE

    # Mean squares + F-test
    MSM   = SSM / p
    MSE   = SSE / (n - p - 1)
    Fstat = MSM / MSE
    pval  = 1 - fdist.cdf(Fstat, p, n - p - 1)

    # Goodness of fit
    r2    = 1 - SSE / SST
    r2adj = 1 - MSE / (SST / (n - 1))
    rmse  = np.sqrt(MSE)
    mae   = np.mean(np.abs(resid))

    return {
        "n": n, "p": p,
        "bhat": bhat,
        "SSE": SSE, "SST": SST, "SSM": SSM,
        "MSE": MSE, "MSM": MSM,
        "Fstat": Fstat, "pval": pval,
        "r2": r2, "r2adj": r2adj,
        "rmse": rmse, "mae": mae,
    }


def print_results(results, label, coef_names=None):
    """Pretty-print the output of fit_ols."""
    print("=" * 60)
    print(f"MODEL: {label}")
    print("=" * 60)
    print(f"n        = {results['n']:,}")
    print(f"p        = {results['p']}")
    print(f"SSE      = {results['SSE']:.2f}")
    print(f"SST      = {results['SST']:.2f}")
    print(f"SSM      = {results['SSM']:.2f}")
    print(f"MSE      = {results['MSE']:.2f}")
    print(f"F-stat   = {results['Fstat']:.4f}")
    print(f"p-value  = {results['pval']:.6e}")
    print(f"R^2      = {results['r2']:.4f}")
    print(f"R^2_adj  = {results['r2adj']:.4f}")
    print(f"RMSE     = {results['rmse']:.2f} sec")
    print(f"MAE      = {results['mae']:.2f} sec")

    if coef_names is not None:
        print("\nCoefficients on numeric predictors:")
        print(f"  {'intercept':<20} {results['bhat'][0]:>12.4f}")
        for name, b in zip(coef_names, results['bhat'][1:1 + len(coef_names)]):
            print(f"  {name:<20} {b:>12.4f}")

## 8. Fit the Full Model

The full specification is:

$$\text{delay\_seconds}_i = \beta_0 + \beta_1 \,\text{pct\_minority}_i + \beta_2 \,\text{pct\_lowincome}_i + \beta_3 \,\text{hour}_i + \beta_4 \,\text{day\_of\_week}_i + \beta_5 \,\text{is\_rush\_hour}_i + \beta_6 \,\text{prev\_delay}_i + \sum_k \gamma_k \,\mathbb{1}[\text{route}_i = k] + \varepsilon_i$$

The route dummies absorb route-specific average reliability differences, so the demographic coefficients $\beta_1, \beta_2$ are estimated **after partialling out** route-level operating conditions.

In [14]:
full_results = fit_ols(X, Y)
print_results(full_results, "Full model (demographics + operating + route)",
              coef_names=numeric_predictors)
# Route dummy coefficients are stored in full_results['bhat'] but not printed
# (too many, and they are nuisance controls).

MODEL: Full model (demographics + operating + route)
n        = 325,924
p        = 41
SSE      = 41166336651.12
SST      = 45200857628.91
SSM      = 4034520977.79
MSE      = 126322.83
F-stat   = 778.9799
p-value  = 1.110223e-16
R^2      = 0.0893
R^2_adj  = 0.0891
RMSE     = 355.42 sec
MAE      = 241.61 sec

Coefficients on numeric predictors:
  intercept                198.1231
  pct_minority            -153.9847
  pct_lowincome             50.6560
  hour                       3.6222
  day_of_week                2.7629
  is_rush_hour              39.4942
  prev_delay                 0.1474


## 9. Comparison: Demographics-Only Model

To quantify how much explanatory power the operating + route controls add, we fit a stripped-down model with only the two demographic predictors. The gap in $R^2$ between this and the full model tells us how much trip-level delay variance is operationally driven vs. demographically associated.

In [15]:
X_demo = trip_df[["pct_minority", "pct_lowincome"]].astype(float).values
demo_results = fit_ols(X_demo, Y)

print_results(demo_results, "Demographics only",
              coef_names=["pct_minority", "pct_lowincome"])

gain = (full_results['r2'] - demo_results['r2']) * 100
print(f"\nR^2 gain from adding operating + route factors: {gain:.2f} percentage points")

MODEL: Demographics only
n        = 325,924
p        = 2
SSE      = 44421629111.24
SST      = 45200857628.91
SSM      = 779228517.68
MSE      = 136295.69
F-stat   = 2858.5955
p-value  = 1.110223e-16
R^2      = 0.0172
R^2_adj  = 0.0172
RMSE     = 369.18 sec
MAE      = 251.55 sec

Coefficients on numeric predictors:
  intercept                712.2272
  pct_minority             -48.2584
  pct_lowincome           -563.4538

R^2 gain from adding operating + route factors: 7.20 percentage points


## 10. Correlation Matrix & Multicollinearity Check (Lecture 16)

Linear regression coefficients become unstable when predictors are highly correlated with each other. We check pairwise correlations among the numeric predictors (and Y) and flag any pair above |r| = 0.7.

The pair to watch is **`pct_minority` vs. `pct_lowincome`** — these often move together at the neighborhood level. If they're collinear, their individual coefficients should be interpreted **jointly** rather than as independent effects.

In [16]:
RR = trip_df[numeric_predictors + [response]].corr()

print("=" * 60)
print("Correlation matrix (numeric predictors + Y)")
print("=" * 60)
print(RR.round(3))

mc = abs(RR.loc["pct_minority", "pct_lowincome"])
print(f"\ncor(pct_minority, pct_lowincome) = {mc:.3f}")

if mc > 0.7:
    print("WARNING: |correlation| > 0.7 - multicollinearity likely.")
    print("  -> Read the two demographic coefficients jointly, not individually.")
else:
    print("OK: below the 0.7 multicollinearity threshold.")

Correlation matrix (numeric predictors + Y)
               pct_minority  pct_lowincome   hour  day_of_week  is_rush_hour  prev_delay  delay_seconds
pct_minority          1.000          0.815 -0.018        0.004        -0.029      -0.113         -0.113
pct_lowincome         0.815          1.000 -0.017        0.008        -0.036      -0.131         -0.131
hour                 -0.018         -0.017  1.000        0.021        -0.060       0.068          0.066
day_of_week           0.004          0.008  0.021        1.000        -0.027       0.001          0.010
is_rush_hour         -0.029         -0.036 -0.060       -0.027         1.000       0.060          0.059
prev_delay           -0.113         -0.131  0.068        0.001         0.060       1.000          0.206
delay_seconds        -0.113         -0.131  0.066        0.010         0.059       0.206          1.000

cor(pct_minority, pct_lowincome) = 0.815
  -> Read the two demographic coefficients jointly, not individually.


## 11. Model Comparison Summary

Side-by-side comparison of the three fits. We're looking for:
1. Demographics-only beats baseline (any signal at all?)
2. Full model beats demographics-only (do operating factors matter?)
3. Whether the full-model RMSE/MAE reduction is practically meaningful, not just statistically significant.

In [17]:
print("=" * 65)
print("MODEL COMPARISON SUMMARY")
print("=" * 65)
print(f"{'Model':<35} {'R^2':>8} {'RMSE':>10} {'MAE':>10}")
print("-" * 65)
print(f"{'Naive baseline (predicts mean)':<35} "
      f"{0.0:>8.4f} {baseline_rmse:>10.2f} {baseline_mae:>10.2f}")
print(f"{'Demographics only':<35} "
      f"{demo_results['r2']:>8.4f} {demo_results['rmse']:>10.2f} "
      f"{demo_results['mae']:>10.2f}")
print(f"{'Full (operating + demos + route)':<35} "
      f"{full_results['r2']:>8.4f} {full_results['rmse']:>10.2f} "
      f"{full_results['mae']:>10.2f}")

MODEL COMPARISON SUMMARY
Model                                    R^2       RMSE        MAE
-----------------------------------------------------------------
Naive baseline (predicts mean)        0.0000     372.40     254.29
Demographics only                     0.0172     369.18     251.55
Full (operating + demos + route)      0.0893     355.42     241.61


## 12. Interpretation

This final cell pulls together the formal hypothesis test and the practical fit improvements:

- **Overall F-test:** Tests $H_0: \beta_1 = \beta_2 = \dots = \beta_p = 0$ (all slopes are zero) against the alternative that at least one is nonzero.
- **R² decomposition:** How much variance does each component explain?
- **Practical improvement:** RMSE/MAE reduction over the naive baseline — what does the model buy us in real-world units (seconds of delay)?

In [18]:
print("=" * 60)
print("INTERPRETATION")
print("=" * 60)

if full_results["pval"] < 0.05:
    print(f"Overall F-test: p = {full_results['pval']:.2e} < 0.05  ->  Reject H0.")
    print("At least one predictor significantly explains trip-level delay.")
else:
    print(f"Overall F-test: p = {full_results['pval']:.2e} >= 0.05  ->  Fail to reject.")

print(f"\nFull model R^2     = {full_results['r2']:.4f}  "
      f"({full_results['r2']*100:.1f}% of trip delay variance)")
print(f"Demographics-only  = {demo_results['r2']:.4f}  "
      f"({demo_results['r2']*100:.1f}%)")
print(f"Operating + route adds {(full_results['r2'] - demo_results['r2'])*100:.2f} pts")

print(f"\nFull model improves over naive baseline by:")
print(f"  RMSE: {baseline_rmse - full_results['rmse']:.2f} sec  "
      f"({(1 - full_results['rmse']/baseline_rmse)*100:.1f}% reduction)")
print(f"  MAE:  {baseline_mae - full_results['mae']:.2f} sec  "
      f"({(1 - full_results['mae']/baseline_mae)*100:.1f}% reduction)")

INTERPRETATION
Overall F-test: p = 1.11e-16 < 0.05  ->  Reject H0.
At least one predictor significantly explains trip-level delay.

Full model R^2     = 0.0893  (8.9% of trip delay variance)
Demographics-only  = 0.0172  (1.7%)
Operating + route adds 7.20 pts

Full model improves over naive baseline by:
  RMSE: 16.99 sec  (4.6% reduction)
  MAE:  12.68 sec  (5.0% reduction)


## Notes & Next Steps

### Caveats
- **OLS assumptions:** linear regression assumes linearity, homoscedasticity, and approximately normal residuals. Trip-level delay data is typically right-skewed (long tail of very late trips), so a residual diagnostic plot would be a worthwhile next step.
- **No train/test split:** the metrics here are in-sample. R² will be optimistic.
- **Aggregation level:** demographics are measured at the route catchment level but joined onto every trip on that route, so the demographic predictors are constant within route. Route fixed effects (the dummies) and demographic coefficients therefore compete for the same between-route variance — the demographic coefficients are identified primarily by between-route differences, not within-route variation.